# Truy vấn - Máy 1 (CN001)


## Khởi tạo kết nối đến cả 2 máy

In [32]:
import pymongo

IP_MAY_1 = '100.80.212.106'
IP_MAY_2 = '100.118.59.95'

client_1 = pymongo.MongoClient(f'mongodb://{IP_MAY_1}:27017/', serverSelectionTimeoutMS=5000)
client_2 = pymongo.MongoClient(f'mongodb://{IP_MAY_2}:27017/', serverSelectionTimeoutMS=5000)

db_1 = client_1['fahasa_btl2']
db_2 = client_2['fahasa_btl2']
print("Đã thiết lập kết nối (client_1: Máy 1, client_2: Máy 2)")

Đã thiết lập kết nối (client_1: Máy 1, client_2: Máy 2)


## 1. Truy vấn trực tiếp tại Máy 1
### Câu truy vấn 1: Phân khúc hoá đơn tại CN001 bằng `$bucket`

In [34]:
pipeline_bucket = [
    {"$bucket": {
        "groupBy": "$TongTien",
        "boundaries": [0, 500000, 2000000, 5000000, 10000000],
        "default": "Lớn hơn 10 triệu",
        "output": {
            "SoLuongHoaDon": {"$sum": 1},
            "TongDoanhThu": {"$sum": "$TongTien"}
        }
    }}
]
print("--- Phân khúc Hóa Đơn tại CN001 ---")
res_bucket = list(db_1.hoadon.aggregate(pipeline_bucket))

ten_phan_khuc = {
    0: "Dưới 500K",
    500000: "Từ 500K - 2 Triệu",
    2000000: "Từ 2 Triệu - 5 Triệu",
    5000000: "Từ 5 Triệu - 10 Triệu",
    "Lớn hơn 10 triệu": "Trên 10 Triệu"
}

for r in res_bucket:
    ten = ten_phan_khuc.get(r['_id'], r['_id'])
    print(f"Phân khúc {ten}: {r['SoLuongHoaDon']} hóa đơn, Tổng thu: {r['TongDoanhThu']:,.0f} VNĐ")

--- Phân khúc Hóa Đơn tại CN001 ---
Phân khúc Dưới 500K: 967 hóa đơn, Tổng thu: 287,425,000 VNĐ
Phân khúc Từ 500K - 2 Triệu: 3539 hóa đơn, Tổng thu: 4,179,375,000 VNĐ
Phân khúc Từ 2 Triệu - 5 Triệu: 516 hóa đơn, Tổng thu: 1,202,439,000 VNĐ


## 2. Máy 1 thực hiện truy vấn đến Máy 2 (Remote)
### Câu truy vấn 2: Dùng `$filter` tìm các hóa đơn tại CN002 có chứa mặt hàng được mua với số lượng sỉ (>= 3 cuốn)

In [35]:
pipeline_filter = [
    {"$project": {
        "_id": 1,
        "MaKH": 1,
        "NgayLap": 1,
        "MatHangSi": {
            "$filter": {
                "input": "$ChiTiet",
                "as": "ct",
                "cond": {"$gte": ["$$ct.SoLuong", 3]}
            }
        }
    }},
    {"$match": {"MatHangSi.0": {"$exists": True}}}, 
    {"$limit": 5}
]

print("--- [Truy vấn Remote] Hóa đơn mua sỉ tại Máy 2 (CN002) ---")
res_filter = list(db_2.hoadon.aggregate(pipeline_filter))
for r in res_filter:
    print(f"Hóa đơn: {r['_id']} | Khách hàng: {r['MaKH']}")
    for mh in r['MatHangSi']:
        print(f"  -> Sách {mh['MaSach']}: {mh['SoLuong']} cuốn")

--- [Truy vấn Remote] Hóa đơn mua sỉ tại Máy 2 (CN002) ---
Hóa đơn: HD00004 | Khách hàng: KH0264
  -> Sách S0080: 3 cuốn
Hóa đơn: HD00005 | Khách hàng: KH0269
  -> Sách S0021: 3 cuốn
Hóa đơn: HD00007 | Khách hàng: KH0226
  -> Sách S0015: 3 cuốn
Hóa đơn: HD00014 | Khách hàng: KH0008
  -> Sách S0043: 3 cuốn
  -> Sách S0093: 3 cuốn
Hóa đơn: HD00019 | Khách hàng: KH0242
  -> Sách S0066: 3 cuốn


## 3. Máy 1 thực hiện truy vấn đến Máy 2 (Local)
### Câu truy vấn 3: Dùng `$unionWith` lập danh bạ chúc mừng sinh nhật tháng 12 (Nhân viên & Khách hàng) tại CN001

In [38]:
pipeline_union = [
    # Luồng 1: Lấy danh sách Nhân viên sinh vào tháng 12
    {"$project": {
        "_id": 0, "Ma": "$_id", "Ten": "$HoTen",
        "VaiTro": {"$literal": "Nhân Viên"},
        "LienHe": "$SoDienThoai",
        "ThangSinh": {"$month": "$NgaySinh"}
    }},
    {"$match": {"ThangSinh": 12}},
    # KẾT HỢP VỚI
    {"$unionWith": {
        "coll": "khachhang",
        "pipeline": [
            # Luồng 2: Lấy danh sách Khách hàng sinh vào tháng 12
            {"$project": {
                "_id": 0, "Ma": "$_id", "Ten": "$HoTen",
                "VaiTro": {"$literal": "Khách Hàng"},
                "LienHe": "$SoDienThoai",
                "ThangSinh": {"$month": "$NgaySinh"}
            }},
            {"$match": {"ThangSinh": 12}}
        ]
    }},
    # Sắp xếp theo tên chữ cái để trộn lẫn kết quả
    {"$sort": {"Ten": 1}},
]

print("--- [Truy vấn Local] Danh bạ chúc mừng sinh nhật tháng 12 (Nhân viên & Khách hàng) tại CN001 ---")
res_union = list(db_1.nhanvien.aggregate(pipeline_union))
for r in res_union:
    print(f"[{r['VaiTro']}] {r['Ten']} - SĐT: {r['LienHe']}")


--- [Truy vấn Local] Danh bạ chúc mừng sinh nhật tháng 12 (Nhân viên & Khách hàng) tại CN001 ---
[Khách Hàng] Bùi Dương Thảo Vy - SĐT: 0149459979
[Nhân Viên] Bùi Quang Minh - SĐT: 0903836205
[Khách Hàng] Bùi Thái Mỹ Linh - SĐT: 0804809418
[Khách Hàng] Bùi Đức Mạnh - SĐT: 0244761543
[Khách Hàng] Bạch Hoàng Minh - SĐT: 0541718499
[Khách Hàng] Hoàng Văn Dũng - SĐT: 0177477398
[Nhân Viên] Lê Thị Ngọc Diện - SĐT: 0551377225
[Khách Hàng] Lê Tuấn Sang - SĐT: 0915562241
[Nhân Viên] Lê Vũ Chương - SĐT: 0163685716
[Khách Hàng] Nguyễn Anh Kiệt - SĐT: 0495516505
[Khách Hàng] Nguyễn Anh Tuấn - SĐT: 0862337984
[Nhân Viên] Nguyễn Minh Hoàng - SĐT: 0161147946
[Khách Hàng] Nguyễn Nhị Lam - SĐT: 0956833116
[Khách Hàng] Nguyễn Phúc Chương - SĐT: 0990553862
[Khách Hàng] Nguyễn Quốc Khánh - SĐT: 0553470836
[Khách Hàng] Nguyễn Thị Gấm - SĐT: 0826381457
[Khách Hàng] Nguyễn Thị Huyền Trân - SĐT: 0156616600
[Khách Hàng] Nguyễn Văn Khánh - SĐT: 0458010437
[Khách Hàng] Nguyễn Văn Trọng - SĐT: 0335894952
[Khách H

## 4. Thực hiện truy vấn cả 2 máy
### Câu truy vấn 4: Thống kê doanh thu theo giới tính và top 3 hoá đơn có giá trị lớn nhất toàn hệ thống (Sử dụng `$facet` kết hợp thuật toán Map-Reduce ở Python)

In [39]:
pipeline_facet = [
    {"$facet": {
        "DoanhThuTheoGioiTinh": [
            {"$lookup": {
                "from": "nhanvien",
                "localField": "MaNV",
                "foreignField": "_id",
                "as": "NV"
            }},
            {"$unwind": "$NV"},
            {"$group": {
                "_id": "$NV.GioiTinh",
                "TongDoanhThu": {"$sum": "$TongTien"}
            }}
        ],
        "TopHoaDonKhung": [
            {"$sort": {"TongTien": -1}},
            {"$limit": 3},
            {"$project": {"MaHD": "$_id", "TongTien": 1, "MaNV": 1, "_id": 0}}
        ]
    }}
]

res_1 = list(db_1.hoadon.aggregate(pipeline_facet))[0]
res_2 = list(db_2.hoadon.aggregate(pipeline_facet))[0]

gop_doanh_thu = {}
for dt in res_1['DoanhThuTheoGioiTinh'] + res_2['DoanhThuTheoGioiTinh']:
    gt = dt['_id']
    gop_doanh_thu[gt] = gop_doanh_thu.get(gt, 0) + dt['TongDoanhThu']

gop_hoa_don = res_1['TopHoaDonKhung'] + res_2['TopHoaDonKhung']

top_hoa_don_global = sorted(gop_hoa_don, key=lambda x: x['TongTien'], reverse=True)[:3]

print("========== BÁO CÁO TỔNG HỢP TOÀN HỆ THỐNG (CN001 + CN002) ==========")
print("\n[1] DOANH THU THEO GIỚI TÍNH NHÂN VIÊN (Tổng cộng trên mọi chi nhánh):")
for gt, tien in gop_doanh_thu.items():
    print(f"   - Nhóm {gt}: {tien:,.0f} VNĐ")

print("\n[2] TOP 3 HÓA ĐƠN CÓ GIÁ TRỊ LỚN NHẤT TOÀN HỆ THỐNG:")
for i, hd in enumerate(top_hoa_don_global, 1):
    print(f"   Top {i}: Hóa đơn {hd['MaHD']} - Trị giá: {hd['TongTien']:,.0f} VNĐ (Lập bởi NV: {hd['MaNV']})")


========== BÁO CÁO TỔNG HỢP TOÀN HỆ THỐNG (CN001 + CN002) ==========

[1] DOANH THU THEO GIỚI TÍNH NHÂN VIÊN (Tổng cộng trên mọi chi nhánh):
   - Nhóm Nữ: 4,386,335,000 VNĐ
   - Nhóm Nam: 6,849,945,000 VNĐ

[2] TOP 3 HÓA ĐƠN CÓ GIÁ TRỊ LỚN NHẤT TOÀN HỆ THỐNG:
   Top 1: Hóa đơn HD01857 - Trị giá: 3,484,000 VNĐ (Lập bởi NV: NV0045)
   Top 2: Hóa đơn HD08759 - Trị giá: 3,451,000 VNĐ (Lập bởi NV: NV0036)
   Top 3: Hóa đơn HD05465 - Trị giá: 3,442,000 VNĐ (Lập bởi NV: NV0036)
